# 009 Validate a Skill End to End

这是第九课：验证这份天气 Skill 是否真的可用。

学习目标：

1. 理解一份 Skill 到了这个阶段，最小验证面应该覆盖什么
2. 学会区分“读起来合理”和“跑起来可靠”
3. 给真实天气 Skill 增加一个最小验证脚本
4. 理解为什么验证脚本应该覆盖主路径、fallback 路径和错误边界

这节课继续使用：

- `.agents/skills/weather-query-assistant/`


## 先明确这节课解决什么问题

到第八课为止，这份天气 Skill 已经有了：

- workflow
- references
- 主路径脚本
- fallback 脚本
- 会话分支

这时候再只靠肉眼看文件，已经不够了。

你需要开始回答一个更现实的问题：

“这份 Skill 改完之后，我怎么快速判断它有没有坏？”


## 一份 Skill 到这个阶段，最少要验证什么

如果把验证范围压到最小，我建议至少覆盖三类东西：

1. 输入标准化
2. 主路径输出
3. fallback 输出和错误边界

也就是说，不必一开始就做复杂测试框架，但至少要把核心链路测一遍。


In [1]:
minimal_validation_surface = {
    'normalize input': True,
    'primary query building': True,
    'fallback query building': True,
    'unsupported fallback location': True,
}

from pprint import pprint
pprint(minimal_validation_surface)


{'fallback query building': True,
 'normalize input': True,
 'primary query building': True,
 'unsupported fallback location': True}


## 先看当前真实目录

这次我们继续扩展同一份真实 Skill，不新造教学专用目录。


In [2]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


In [3]:
def print_tree(root: Path, prefix: str = '') -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for index, entry in enumerate(entries):
        connector = '└── ' if index == len(entries) - 1 else '├── '
        print(prefix + connector + entry.name)
        if entry.is_dir():
            next_prefix = prefix + ('    ' if index == len(entries) - 1 else '│   ')
            print_tree(entry, next_prefix)


print(skill_root)
print_tree(skill_root)


.agents/skills/weather-query-assistant
├── agents
│   └── openai.yaml
├── references
│   └── weather_sources.md
├── scripts
│   ├── __pycache__
│   │   ├── build_open_meteo_query.cpython-310.pyc
│   │   ├── build_wttr_query.cpython-310.pyc
│   │   ├── normalize_location.cpython-310.pyc
│   │   └── normalize_location.cpython-313.pyc
│   ├── build_open_meteo_query.py
│   ├── build_wttr_query.py
│   ├── normalize_location.py
│   └── validate_weather_skill.py
└── SKILL.md


## 先看 `SKILL.md` 的变化

这次你应该注意到：

- `Scripts` 小节里新增了 `validate_weather_skill.py`

这意味着验证也已经成为这份 Skill 的正式组成部分，而不是临时的手工操作。


In [4]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Normalize the location when the user input is noisy or inconsistently formatted.
4. Build a stable weather query string before calling the external service.
5. Use `wttr.in` as the primary source.
6. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
7. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
8. Do not guess when weather data is unavailable.

## References

- Read `referenc

## 读真实验证脚本

这份脚本的设计目标不是替代完整测试框架，而是提供一条快速健康检查路径。

你应该重点观察：

1. 它覆盖了哪些检查点
2. 它有没有覆盖失败分支
3. 它的输出是否足够直接


In [5]:
print((skill_root / 'scripts' / 'validate_weather_skill.py').read_text(encoding='utf-8'))


#!/usr/bin/env python3
from build_open_meteo_query import build_open_meteo_query
from build_wttr_query import build_wttr_query
from normalize_location import normalize_location


def main() -> int:
    checks = [
        ("normalize beijing", normalize_location("beijing"), "Beijing"),
        ("normalize new york", normalize_location(" New   York "), "New+York"),
        ("normalize airport code", normalize_location("jfk"), "JFK"),
        (
            "build wttr current",
            build_wttr_query("beijing", mode="current"),
            "https://wttr.in/Beijing?m&format=3",
        ),
        (
            "build wttr compact uscs",
            build_wttr_query("jfk", mode="compact", units="uscs"),
            "https://wttr.in/JFK?u&format=%l:+%c+%t+%h+%w",
        ),
        (
            "build open-meteo beijing",
            build_open_meteo_query("beijing"),
            "https://api.open-meteo.com/v1/forecast?latitude=39.9042&longitude=116.4074&current_weather=true",
       

## 直接跑一遍验证脚本

正式开发里，验证脚本的价值不在于“存在”，而在于你真的能跑它。


In [6]:
import subprocess

result = subprocess.run(
    ['python', '.agents/skills/weather-query-assistant/scripts/validate_weather_skill.py'],
    capture_output=True,
    text=True,
    check=False,
)

print(result.stdout)
print('returncode =', result.returncode)


[OK] normalize beijing
[OK] normalize new york
[OK] normalize airport code
[OK] build wttr current
[OK] build wttr compact uscs
[OK] build open-meteo beijing
[OK] unsupported fallback location should raise
validation-ok

returncode = 0


## 这份验证脚本到底覆盖了什么

从验证面上看，它已经覆盖了：

- `normalize_location()`
- `build_wttr_query()`
- `build_open_meteo_query()`
- 不支持地点时的异常分支

这已经足够支撑一份教学型 Skill 的快速回归检查。


In [7]:
validation_coverage = [
    'normalized location output',
    'primary wttr.in URL output',
    'fallback Open-Meteo URL output',
    'unsupported fallback location error path',
]

pprint(validation_coverage)


['normalized location output',
 'primary wttr.in URL output',
 'fallback Open-Meteo URL output',
 'unsupported fallback location error path']


## 为什么这里先不用完整测试框架

你当然可以继续往后加：

- `pytest`
- fixture
- snapshot
- CI

但对当前这份教学型 Skill 来说，这一步太早了。

这节课的重点是先建立验证习惯，而不是先上完整测试体系。

也就是说，这里的验证脚本更像“轻量级健康检查”。


In [8]:
why_not_full_test_framework_yet = [
    'goal is habit building first',
    'repo-local skill is still a teaching artifact',
    'fast feedback matters more than framework completeness at this stage',
]

pprint(why_not_full_test_framework_yet)


['goal is habit building first',
 'repo-local skill is still a teaching artifact',
 'fast feedback matters more than framework completeness at this stage']


## 如果未来要继续工程化，可以往哪边扩展

当这份 Skill 不再只是教学，而是开始承担更长期的维护责任时，后面自然可以扩展：

1. 把验证脚本拆成更细的测试
2. 引入正式测试框架
3. 把验证纳入自动化流程

但这些都应该建立在“已经有一条可靠的最小验证路径”之上。


In [9]:
future_validation_expansion = {
    'next': ['split checks into finer tests', 'use a formal test framework', 'add automation'],
    'base_required_first': 'keep a simple reliable validation path',
}

pprint(future_validation_expansion)


{'base_required_first': 'keep a simple reliable validation path',
 'next': ['split checks into finer tests',
          'use a formal test framework',
          'add automation']}


## 现在这条天气 Skill 教学线还剩多少

按我们之前规划的总路线：

- 总共建议 10 课

现在已经完成：

1. 读真实 Skill
2. 新建最小 Skill
3. 引入 references
4. 引入 scripts
5. 接成 mini workflow
6. 跑完整小会话
7. 处理错误和 fallback
8. 实现真实 Open-Meteo fallback
9. 建立最小验证路径

所以只剩最后 1 课：

10. 完整收尾与复盘


In [10]:
course_progress = {
    'total_recommended': 10,
    'completed_after_this_lesson': 9,
    'remaining': 1,
    'last_lesson': '010 final recap: how the weather skill grew from 0 to usable',
}

pprint(course_progress)


{'completed_after_this_lesson': 9,
 'last_lesson': '010 final recap: how the weather skill grew from 0 to usable',
 'remaining': 1,
 'total_recommended': 10}


## 当前阶段结论

你现在需要记住：

1. Skill 发展到这个阶段，最少应该有一条可运行的验证路径
2. 验证脚本的目标不是炫技，而是快速回答“这份 Skill 现在是不是还正常”
3. 主路径、fallback 路径和错误边界，都是验证面的一部分
4. `validate_weather_skill.py` 让这份天气 Skill 第一次具备了系统性的快速回归检查能力
5. Skill 教学线到这里已经只剩最后一课：完整复盘

下一步建议：

- 继续第十课：完整复盘这份天气 Skill 是怎么从 0 长到可用的
